In [ ]:
# %% [1] — Imports
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import importlib

import solenoid_lib
importlib.reload(solenoid_lib)

from solenoid_lib import (
    solenoid_length,
    theta_solenoid,
    w_tape_mm,
    t_tape_mm,
    mu0,
    Ic_tape,
    Je_tape,
    solenoid_field_center,
    solenoid_field_profile,
    magnetic_energy,
    hoop_stress,
    solenoid_summary,
    b_peak,
)

# ─────────────────────────────────────────────────────────────────────────────
# Parameters
# ─────────────────────────────────────────────────────────────────────────────
je_fixed    = 15.31733963        # [A/mm²]  fixed operating current density
je_fixed_si = je_fixed * 1e6  # [A/m²]   SI version for solenoid functions

ri = 1 # [m] inner radius of solenoid
rf = 1.780241812  # [m] outer radius of solenoid
solenoid_length = 9  # [m] length of solenoid

b0 = solenoid_field_center(ri, rf, je_fixed_si, solenoid_length)
b_peak_test=b_peak(ri, rf, je_fixed_si, solenoid_length)

sigma_pa, b = hoop_stress(ri, rf, je_fixed_si, solenoid_length)


Ic = Ic_tape(b0, theta_solenoid)   # [A/mm²]

j_crit_mm2,je_max_mm2 = Je_tape(b0, theta_solenoid)   # [A/mm²]

j_crit_mm2_peak,je_max_mm2_peak = Je_tape(b_peak_test, theta_solenoid)   # [A/mm²]

print(f"  B0 at je_fiexd           = {b0:.0f} T")

print(f"  B0           = {b:.4f} T")
print(f"  Ic           = {Ic:.0f} A")

print(f"  Bpeak           = {b_peak_test:.4f} T")

print(f"  hoop stress @ B0           = {sigma_pa/1e6:.0f} MPa")
#print(f"  magnetic energy @ B0           = {E_mag/1e6:.0f} MJ")
print(f"  Jc @ B0           = {j_crit_mm2:.0f} A")
print(f"  Je @ B0           = {je_max_mm2:.0f} A")

print(f"  Jc @ Bpeak           = {j_crit_mm2_peak:.0f} A")
print(f"  Je @ Bpeak           = {je_max_mm2_peak:.0f} A")

  B0 at je_fiexd           = 14 T
  B           = 14.3372 T
  Ic           = 1359 A
  B           = 14.3776 T
  hoop stress @ B0           = 208 MPa
  Jc @ B0           = 3397 A
  Je @ B0           = 2038 A
  Jc @ Bpeak           = 3393 A
  Je @ Bpeak           = 2036 A


In [86]:
"""
Solenoid summary: peak conductor field, central field, and hoop stress
by thin-shell magnetic pressure and by Wilson thick wall.
"""

import math

MU0 = 4.0e-7 * math.pi
P0 = {0: 1.0, 2: -1/2, 4: 3/8, 6: -5/16, 8: 35/128, 10: -63/256}   # P_n(0)


def _br(g, a, b):                      # [ ... ]_{r=1}^{r=alpha}
    return g(a, b) - g(1.0, b)

def F(a, b):
    return b * math.log((a + math.hypot(a, b)) / (1.0 + math.hypot(1.0, b)))

def FE2(a, b):
    g = lambda r, b: r**3 / (r*r + b*b)**1.5
    return -_br(g, a, b) / (2 * b)

def FE4(a, b):
    g = lambda r, b: r**3*(2*r**4 + 7*r**2*b**2 + 20*b**4) / (r*r + b*b)**3.5
    return -_br(g, a, b) / (24 * b**3)

def FE6(a, b):
    g = lambda r, b: r**3*(8*r**8 + 44*r**6*b**2 + 99*r**4*b**4
                           + 28*r**2*b**6 + 280*b**8) / (r*r + b*b)**5.5
    return -_br(g, a, b) / (240 * b**5)

def FE8(a, b):
    g = lambda r, b: r**3*(16*r**12 + 120*r**10*b**2 + 390*r**8*b**4 + 715*r**6*b**6
                           + 1080*r**4*b**8 - 1008*r**2*b**10
                           + 1344*b**12) / (r*r + b*b)**7.5
    return -_br(g, a, b) / (896 * b**7)

def FE10(a, b):
    g = lambda r, b: r**3*(128*r**16 + 1216*r**14*b**2 + 5168*r**12*b**4
                           + 12920*r**10*b**6 + 20995*r**8*b**8 + 19976*r**6*b**10
                           + 49632*r**4*b**12 - 46464*r**2*b**14
                           + 21120*b**16) / (r*r + b*b)**9.5
    return -_br(g, a, b) / (11520 * b**9)

TERMS = {0: F, 2: FE2, 4: FE4, 6: FE6, 8: FE8, 10: FE10}


# ---------------------------------------------------------------- fields
def b_peak(length, r_inner, r_outer, j, nmax=10):
    """Peak conductor field at (r,z) = (a1,0), Legendre expansion at xi = 1. [T]"""
    a1 = r_inner
    alpha, beta = r_outer / a1, 0.5 * length / a1
    return MU0 * j * a1 * sum(P0[n] * TERMS[n](alpha, beta)
                              for n in sorted(TERMS) if n <= nmax)


def b_center(length, r_inner, r_outer, j):
    """On-axis central field, exact Biot-Savart over the winding cross-section. [T]"""
    zp, zm = 0.5 * length, -0.5 * length

    def log_term(z):
        return z * math.log((math.hypot(r_outer, z) + r_outer) /
                            (math.hypot(r_inner, z) + r_inner))

    return 0.5 * MU0 * j * (log_term(zp) - log_term(zm))


# ---------------------------------------------------------------- stresses
def wilson_peak(length, r_inner, r_outer, j, B1, nu=0.3, kappa=0.0, npts=201):
    """Peak Wilson hoop stress [Pa] at the bore, for a given B1."""
    a1, alpha = r_inner, r_outer / r_inner
    S = j * B1 * a1 / (alpha - 1.0)

    kA = (2 + nu) / 3 * (alpha - kappa)
    kB = (3 + nu) / 8 * (1 - kappa)

    C0 = kA * (alpha**2 + alpha + 1) / (alpha + 1) - kB * (alpha**2 + 1)
    C2 = alpha**2 * (kA / (alpha + 1) - kB)
    C1 = -(1 + 2*nu) / 3 * (alpha - kappa)
    C3 = (1 + 3*nu) / 8 * (1 - kappa)

    return max(S * (C0 + C2 / p**2 + C1 * p + C3 * p**2)
               for p in (1.0 + (alpha - 1.0) * i / (npts - 1) for i in range(npts)))


def summary(length, r_inner, r_outer, j, nu=0.3, kappa=0.0):
    """
    B1     peak conductor field at (a1,0), Legendre expansion   [T]
    B0     central field on axis, Biot-Savart                   [T]
    sig_p  thin-shell magnetic pressure (B0^2/2mu0)(a1/th)      [Pa]
    sig_w  Wilson peak hoop at the bore, using B1               [Pa]
    """
    th = r_outer - r_inner
    B1 = b_peak(length, r_inner, r_outer, j)
    B0 = b_center(length, r_inner, r_outer, j)
    sig_p = (B0**2 / (2.0 * MU0)) * (r_inner / th)
    sig_w = wilson_peak(length, r_inner, r_outer, j, B1, nu, kappa)
    je_ref    = 1e8          # [A/m²]  arbitrary reference (100 A/mm²)
    sigma_ref =750e6       # [Pa]    arbitrary reference (750 MPa)
    # σ ∝ Je²  →  Je_lim = Je_ref * sqrt(sigma_limit / sigma_ref)
    je_lim = je_ref * np.sqrt(sig_p / sigma_ref)


    print(f"\nL = {length*1e3:.1f} mm   a1 = {r_inner*1e3:.1f} mm   "
          f"a2 = {r_outer*1e3:.1f} mm   j = {j/1e6:.1f} A/mm^2")
    print(f"B1  (peak, Legendre)         = {B1:8.4f} T")
    print(f"B0  (centre, Biot-Savart)    = {B0:8.4f} T")
    print(f"sigma_hoop (mag. pressure)   = {sig_p/1e6:8.2f} MPa")
    print(f"sigma_hoop (Wilson, peak)    = {sig_w/1e6:8.2f} MPa")
    print(f"Je_lim (σ_ref = 750 MPa)     = {je_lim/1e6:8.2f} A/mm^2")
    return B1, B0, sig_p, sig_w


summary(length=9, r_inner=1.5, r_outer=1.780241812, j=15.31733963e6)


L = 9000.0 mm   a1 = 1500.0 mm   a2 = 1780.2 mm   j = 15.3 A/mm^2
B1  (peak, Legendre)         =   5.1073 T
B0  (centre, Biot-Savart)    =   5.0676 T
sigma_hoop (mag. pressure)   =    54.69 MPa
sigma_hoop (Wilson, peak)    =    70.36 MPa
Je_lim (σ_ref = 750 MPa)     =    27.00 A/mm^2


(5.107345981346422, 5.0675940601610385, 54691785.75010316, 70359392.20016949)

In [104]:
import numpy as np
import solenoid_lib

# ═════════════════════════════════════════════════════════════════════
# INPUTS
# ═════════════════════════════════════════════════════════════════════
ri          = 1          # inner radius [m]
th          = 0.02        # radial build [m]
L_req       = 5           # requested length [m]

sigma_limit = 750e6         # hoop limit, same for tape substrate and filler [Pa]
fCu         = 0.50          # copper fraction of the tape itself [-]
gamma_cu    = 5.0e16        # copper action limit at the hot spot [A2 s m-4]

n_par       = 10            # tapes co-wound radially, paralleled into one turn
U_EE = 1000.0          # dump-circuit terminal voltage [V]

theta     = solenoid_lib.theta_solenoid
gamma_max = gamma_cu * fCu

# ── fixed geometry: axial pitch is the tape width, so L quantises exactly ────
t_tape = solenoid_lib.t_tape_mm * 1e-3
w_tape = solenoid_lib.w_tape_mm * 1e-3
a_tape = t_tape * w_tape
n_pc   = max(1, round(L_req / w_tape))
L      = n_pc * w_tape
rf     = ri + th
a_wind = th * L
v_bore = np.pi * ri ** 2 * L

# ── power-law coefficients, evaluated once at Je = 1 A/mm2 ──────────────────
# B0 = K_B*Je, sigma = K_S*Je^2, E = K_E*Je^2 with Je in A/mm2. These are the
# same three library calls the old code made on every probe; the exponents are
# exact, which is what the scaling guard at the bottom verifies.
K_B = solenoid_lib.solenoid_field_center(ri, rf, 1e6, L)
K_S = solenoid_lib.hoop_stress(ri, rf, 1e6, L)[0]
K_E = solenoid_lib.magnetic_energy(ri, rf, 1e6, L)


# ═════════════════════════════════════════════════════════════════════
# EVALUATION — split in two.
#   r_of()   : lean predicate for the search, returns the scalar r only.
#   report() : same chain, full dict, called twice at the end.
# The chains must stay identical; an assert below pins them together.
# ═════════════════════════════════════════════════════════════════════
def r_of(je_mm2):
    je = je_mm2 * 1e6

    b0 = K_B * je_mm2
    _, je_tape_mm2 = solenoid_lib.Je_tape(b0, theta)
    if not np.isfinite(je_tape_mm2) or je_tape_mm2 <= 0.0:
        return np.inf

    f_tape   = je_mm2 / je_tape_mm2
    n_tp_min = int(np.ceil(f_tape * th / t_tape))
    n_bund   = max(1, int(np.ceil(n_tp_min / n_par)))
    n_tot    = n_bund * n_pc
    f_built  = (n_par * n_bund * n_pc) * a_tape / a_wind
    i0       = je * a_wind / n_tot

    em_tot = K_E * je_mm2 ** 2
    l_tot  = 2.0 * em_tot / i0 ** 2
    r_ee   = U_EE / i0
    tau_ee = l_tot / r_ee

    j_cu_max  = np.sqrt(gamma_max / (0.5 * tau_ee))
    f_cu_req  = je / j_cu_max
    f_cu_have = f_built * fCu
    f_cu_add  = max(f_cu_req - f_cu_have, 0.0)
    f_cu_eff  = max(f_cu_req, f_cu_have)

    sigma_pa     = K_S * je_mm2 ** 2
    f_struct     = 1.0 - f_cu_eff
    sigma_struct = sigma_pa / f_struct if f_struct > 0.0 else np.inf
    util         = sigma_struct / sigma_limit
    fill         = f_built + f_cu_add
    return max(util, fill)


def report(je_mm2):

    d  = {"je": je_mm2}
    je = je_mm2 * 1e6

    # ── EM ───────────────────────────────────────────────────────────
    b0             = K_B * je_mm2
    sigma_pa       = K_S * je_mm2 ** 2
    _, je_tape_mm2 = solenoid_lib.Je_tape(b0, theta)
    d.update(b0=b0, sigma_pa=sigma_pa, je_tape=je_tape_mm2,
             scan=solenoid_lib.scan_time(b0, v_bore))

    if not np.isfinite(je_tape_mm2) or je_tape_mm2 <= 0.0:
        d.update(r=np.inf, binding="EM/tape")
        return d                                   # tape carries nothing here

    # ── winding layout: n_par tapes stacked radially = one electrical turn ──
    f_tape   = je_mm2 / je_tape_mm2
    n_tp_min = int(np.ceil(f_tape * th / t_tape))       # tapes needed radially
    n_bund   = max(1, int(np.ceil(n_tp_min / n_par)))   # bundles per pancake
    n_tp     = n_par * n_bund                           # tapes per pancake
    n_turn_p = n_bund                                   # turns per pancake
    n_tape   = n_tp * n_pc                              # physical tapes, total
    n_tot    = n_turn_p * n_pc                          # electrical turns, total
    f_built  = n_tape * a_tape / a_wind
    i0       = je * a_wind / n_tot                      # terminal current [A]
    d.update(f_tape=f_tape, n_tp_min=n_tp_min, n_tp=n_tp, n_turn_p=n_turn_p,
             n_tape=n_tape, n_tot=n_tot, f_built=f_built, i0=i0,
             len_tape=n_tape * np.pi * (ri + rf))

    # ── inductance ───────────────────────────────────────────────────
    # L = 2E/I^2 with I the terminal current. Grouping n_par tapes into one turn
    # leaves em_tot untouched and multiplies i0 by n_par, so l_tot falls as
    # 1/n_par^2. One resistor for the whole stack: the dump sees l_tot, em_tot.
    em_tot = K_E * je_mm2 ** 2
    l_tot  = 2.0 * em_tot / i0 ** 2

    # ── dump circuit ─────────────────────────────────────────────────
    u_ee = 1000 
    r_ee   = u_ee/i0
    tau_ee = l_tot / r_ee

    print(r_ee)
    print(tau_ee)

    d.update(em_tot=em_tot, l_tot=l_tot, r_ee=r_ee, tau_ee=tau_ee, u_ee=u_ee)

    # ── hot spot, adiabatic: exponential dump only ───────────────────
    # Smeared over the build j_cu = Je/f_cu, so the turn count cancels here.
    j_cu_max  = np.sqrt(gamma_max / (0.5 * tau_ee))
    f_cu_req  = je / j_cu_max                      # total Cu fraction of build
    f_cu_have = f_built * fCu                      # Cu already inside the tape
    f_cu_add  = max(f_cu_req - f_cu_have, 0.0)     # co-wound Cu to add
    f_cu_eff  = max(f_cu_req, f_cu_have)           # Cu actually in the build
    d.update(j_cu_max=j_cu_max, f_cu_req=f_cu_req, f_cu_add=f_cu_add,
             f_cu_eff=f_cu_eff,
             gamma_op=(je / f_cu_eff) ** 2 * (0.5 * tau_ee))

    assert d["gamma_op"] <= gamma_max * (1 + 1e-9), "hot-spot sizing inconsistent"

    # ── mechanics: copper carries nothing (worst case) ───────────────
    # Ri, Th, L are fixed, so copper changes neither the Lorentz load nor
    # sigma_hoop. It changes what is left to carry it. Everything non-Cu is
    # load-bearing at sigma_limit: the tape substrate and the filler alike.
    f_struct      = 1.0 - f_cu_eff                 # all non-Cu material
    f_struct_tape = f_built * (1.0 - fCu)          # non-Cu part of the tape
    f_filler      = f_struct - f_struct_tape       # added structural material
    sigma_struct  = sigma_pa / f_struct if f_struct > 0.0 else np.inf
    sigma_tape    = (sigma_pa / f_struct_tape      # if the tape carried it alone
                     if f_struct_tape > 0.0 else np.inf)
    util          = sigma_struct / sigma_limit
    fill          = f_built + f_cu_add
    d.update(f_struct=f_struct, f_struct_tape=f_struct_tape, f_filler=f_filler,
             sigma_struct=sigma_struct, sigma_tape=sigma_tape,
             util=util, fill=fill)

    # ── binding constraint ───────────────────────────────────────────
    d["r"] = max(util, fill)
    d["binding"] = ("EM/tape" if f_tape > 1.0 else
                    "mechanical" if util >= fill else "packing")
    return d


# ═════════════════════════════════════════════════════════════════════
# SOLVE — r(Je) is monotone increasing (jumps at each new layer go UP),
#         so the feasible set is (0, Je*] and bisection is exact.
# ═════════════════════════════════════════════════════════════════════
RTOL = 1e-9           # relative width of the final bracket

# all of Th as structure, zero copper -> infeasible by construction
je_seed = solenoid_lib.je_max_stress_limited(ri, rf, L, sigma_limit) / 1e6




je_hi   = je_seed
n_ev    = 0


def probe(je):
    global n_ev
    n_ev += 1
    return r_of(je)


while probe(je_hi) <= 1.0:                    # only if quench costs nothing
    je_hi *= 2.0

je_lo = None                                  # lower bound: halve until it fits
je = je_hi
for _ in range(200):
    je *= 0.5
    if probe(je) <= 1.0:
        je_lo = je
        break
    je_hi = je
if je_lo is None:
    raise RuntimeError("infeasible for every Je: check gamma_max, R_EE, sigma_limit")

while je_hi - je_lo > RTOL * je_lo:
    je_mid = 0.5 * (je_lo + je_hi)
    if probe(je_mid) <= 1.0:
        je_lo = je_mid
    else:
        je_hi = je_mid

d = report(je_lo)
assert abs(d["r"] - r_of(je_lo)) < 1e-12, "r_of and report have diverged"

print(f"converged: {n_ev} evaluations, Je* = {d['je']:.4f} A/mm2 "
      f"(bracket {je_hi - je_lo:.2e}), binding = {d['binding']}, "
      f"fill = {d['fill']:.4f}, n_tp = {d['n_tp']}")
print(f"  budget: tape {d['f_built']*100:5.2f} %  Cu added {d['f_cu_add']*100:5.2f} % "
      f"|  Cu total {d['f_cu_req']*100:5.2f} %, "
      f"load-bearing {d['f_struct']*100:5.2f} % "
      f"(tape {d['f_struct_tape']*100:.2f} % + filler {d['f_filler']*100:.2f} %)")


# ═════════════════════════════════════════════════════════════════════
# FINAL MAGNET — converged design vs. the stress-limited starting guess.
# The seed column is infeasible by construction (sigma_struct > sigma_limit):
# it is the no-quench-protection limit, not a buildable alternative.
# ═════════════════════════════════════════════════════════════════════
d0 = report(je_seed)
g  = lambda k: d0.get(k, np.nan)

rows = [
    ("── geometry (fixed) ──", "", None, None, None),
    ("Ri",                 "mm",    ri * 1e3,            ri * 1e3,            "12.1f"),
    ("Rf",                 "mm",    rf * 1e3,            rf * 1e3,            "12.1f"),
    ("Th",                 "mm",    th * 1e3,            th * 1e3,            "12.3f"),
    ("length",             "mm",    L * 1e3,             L * 1e3,             "12.1f"),

    ("── operating point ──", "", None, None, None),
    ("Je",                 "A/mm2", g("je"),             d["je"],             "12.2f"),
    ("B0 centre",          "T",     g("b0"),             d["b0"],             "12.3f"),
    ("scan time",          "yr",    g("scan"),           d["scan"],           "12.4g"),
    ("E stored",           "MJ",    g("em_tot") / 1e6,   d["em_tot"] / 1e6,   "12.2f"),
    ("L self",             "H",     g("l_tot"),          d["l_tot"],          "12.3f"),

    ("── winding ──", "", None, None, None),
    ("tapes in parallel",  "-",     n_par,               n_par,               "12.0f"),
    ("tapes / pancake min","-",     g("n_tp_min"),       d["n_tp_min"],       "12.0f"),
    ("tapes / pancake",    "-",     g("n_tp"),           d["n_tp"],           "12.0f"),
    ("turns / pancake",    "-",     g("n_turn_p"),       d["n_turn_p"],       "12.0f"),
    ("pancakes",           "-",     n_pc,                n_pc,                "12.0f"),
    ("tapes total",        "-",     g("n_tape"),         d["n_tape"],         "12.0f"),
    ("turns total",        "-",     g("n_tot"),          d["n_tot"],          "12.0f"),
    ("I terminal (turn)",  "A",     g("i0"),             d["i0"],             "12.1f"),
    ("Je tape @B0",        "A/mm2", g("je_tape"),        d["je_tape"],        "12.2f"),
    ("J tape as built",    "A/mm2", g("je") / g("f_built"),
                                    d["je"] / d["f_built"],                   "12.2f"),
    ("f_tape needed",      "%",     g("f_tape") * 100,   d["f_tape"] * 100,   "12.2f"),
    ("f_tape built",       "%",     g("f_built") * 100,  d["f_built"] * 100,  "12.2f"),
    ("tape length",        "km",    g("len_tape") / 1e3, d["len_tape"] / 1e3, "12.2f"),

    ("── quench ──", "", None, None, None),
    ("R_EE",               "ohm",   g("r_ee"),           d["r_ee"],           "12.3f"),
    ("U_EE",               "V",     g("u_ee"),           d["u_ee"],           "12.1f"),
    ("tau_EE",             "s",     g("tau_ee"),         d["tau_ee"],         "12.4f"),
    ("J_Cu allowed",       "A/mm2", g("j_cu_max") / 1e6, d["j_cu_max"] / 1e6, "12.1f"),
    ("f_Cu required",      "%",     g("f_cu_req") * 100, d["f_cu_req"] * 100, "12.2f"),
    ("f_Cu co-wound",      "%",     g("f_cu_add") * 100, d["f_cu_add"] * 100, "12.2f"),

    ("── mechanics ──", "", None, None, None),
    ("sigma_hoop smeared", "MPa",   g("sigma_pa") / 1e6, d["sigma_pa"] / 1e6, "12.1f"),
    ("f_structural",       "%",     g("f_struct") * 100, d["f_struct"] * 100, "12.2f"),
    ("  of which tape",    "%",     g("f_struct_tape") * 100,
                                    d["f_struct_tape"] * 100,                 "12.2f"),
    ("  of which filler",  "%",     g("f_filler") * 100, d["f_filler"] * 100, "12.2f"),
    ("sigma tape alone",   "MPa",   g("sigma_tape") / 1e6,
                                    d["sigma_tape"] / 1e6,                    "12.1f"),
    ("sigma structure",    "MPa",   g("sigma_struct") / 1e6,
                                    d["sigma_struct"] / 1e6,                  "12.1f"),
    ("sigma limit",        "MPa",   sigma_limit / 1e6,   sigma_limit / 1e6,   "12.1f"),
    ("packing",            "%",     g("fill") * 100,     d["fill"] * 100,     "12.2f"),
    ("binding",            "-",     g("binding"),        d["binding"],        "s"),
]

w = 24 + 9 + 12 + 12 + 10
print(f"\n{'FINAL MAGNET':<24}{'unit':<9}{'seed':>12}{'converged':>12}{'delta %':>10}")
print("=" * w)
for label, unit, v0, v1, fmt in rows:
    if fmt is None:
        print(f"{label}")
        continue
    if fmt == "s":
        print(f"{label:<24}{unit:<9}{str(v0):>12}{str(v1):>12}{'':>10}")
        continue
    ok0 = v0 is not None and np.isfinite(v0)
    ok1 = v1 is not None and np.isfinite(v1)
    s0 = f"{v0:{fmt}}" if ok0 else f"{'-':>12}"
    s1 = f"{v1:{fmt}}" if ok1 else f"{'-':>12}"
    s2 = (f"{(v1 / v0 - 1.0) * 100:10.2f}"
          if (ok0 and ok1 and v0 != 0.0 and v0 != v1) else f"{'-':>10}")
    print(f"{label:<24}{unit:<9}{s0}{s1}{s2}")
print("=" * w)

# ── tape-only check: can the substrate carry the load without filler? ──
print(f"  tape alone : {d['sigma_tape']/1e6:.0f} MPa vs {sigma_limit/1e6:.0f} MPa "
      f"limit -> {'no filler needed' if d['sigma_tape'] <= sigma_limit else 'needs filler'}"
      f", {d['f_filler']*th*1e3:.1f} mm of the {th*1e3:.0f} mm build")

# ── regression guard: exact by construction inside solenoid_lib ─────
dj = d["je"] / g("je") - 1.0
db = d["b0"] / g("b0") - 1.0
ds = d["sigma_pa"] / g("sigma_pa") - 1.0
scaling_ok = abs(db - dj) < 1e-6 and abs(ds - ((1 + dj) ** 2 - 1)) < 1e-6
print(f"  scaling B0~Je, sigma~Je^2 : {'ok' if scaling_ok else 'MISMATCH'}")
print(f"  verdict : {'PASS' if (d['util'] <= 1.0 and d['fill'] <= 1.0) else 'FAIL'}, "
      f"limited by {d['binding']}, {d['je']:.2f} A/mm2 at {d['b0']:.3f} T")

0.14197826842910707
4.9444746559333534
converged: 33 evaluations, Je* = 88.0416 A/mm2 (bracket 5.81e-08), binding = mechanical, fill = 0.9005, n_tp = 10
  budget: tape  5.00 %  Cu added 85.05 % |  Cu total 87.55 %, load-bearing 12.45 % (tape 2.50 % + filler 9.95 %)
0.10018707822371928
28.027883930238108

FINAL MAGNET            unit             seed   converged   delta %
── geometry (fixed) ──
Ri                      mm             1000.0      1000.0         -
Rf                      mm             1020.0      1020.0         -
Th                      mm             20.000      20.000         -
length                  mm             5000.0      5000.0         -
── operating point ──
Je                      A/mm2          249.53       88.04    -64.72
B0 centre               T               5.815       2.052    -64.72
scan time               yr              76.44        4933   6352.98
E stored                MJ             139.88       17.41    -87.55
L self                  H            